# **KKBOX Churn Prediction And Retention Intelligence System - Streamlit integration (App Setting)**

---

## **1. Objective**

This notebook integrates the trained churn prediction model into an interactive Streamlit interface, enabling real‑time customer scoring and retention insights.

The goal is to:

- Build a user‑friendly UI for entering customer attributes

- Validate inputs and ensure compatibility with the trained model

- Generate churn probability, risk level, and business impact metrics

- Provide explainability, timing estimates, and rule‑based churn drivers

- Prepare a complete scoring pipeline ready for deployment in the Retention OS

At this stage, the focus is on model inference, UI integration, and interactive prediction.
The outcome is a fully functional Streamlit prototype that demonstrates how the churn model behaves with real‑time user inputs.

---

# **2. UI CONFIG (Dropdowns, Sliders, Defaults)**

In [1]:
import streamlit as st

def ui_customer_input():
    st.sidebar.title("Customer Profile")

    gender = st.sidebar.selectbox("Gender", ["male", "female", "unknown"], index=0)

    age = st.sidebar.slider("Age", 0, 100, 30)

    city_grouped = st.sidebar.selectbox(
        "City Grouped",
        ["1","2","3","4","5","6","7","8","9","10","other"],
        index=4
    )

    registered_via_grouped = st.sidebar.selectbox(
        "Registered Via Grouped",
        ["3","7","9","13","other"],
        index=0
    )

    listening_group = st.sidebar.selectbox(
        "Listening Group",
        ["Low", "Medium-Low", "Medium-High", "High"],
        index=1
    )

    has_auto_renew = st.sidebar.selectbox("Has Auto-Renew", [0, 1], index=1)
    has_cancelled = st.sidebar.selectbox("Has Cancelled", [0, 1], index=0)

    avg_amount_paid = st.sidebar.slider("Avg Amount Paid", 0, 2000, 120)
    total_amount_paid = st.sidebar.slider("Total Amount Paid", 0, 20000, 600)

    total_secs = st.sidebar.slider("Total Listening Seconds", 0, 2_000_000, 50_000)
    num_unq = st.sidebar.slider("Unique Songs", 0, 20000, 200)

    customer_tenure_days = st.sidebar.slider("Customer Tenure (days)", 0, 5000, 800)
    payment_variability = st.sidebar.slider("Payment Variability", 0, 500, 20)

    return {
        "gender": gender,
        "age": age,
        "city_grouped": city_grouped,
        "registered_via_grouped": registered_via_grouped,
        "avg_amount_paid": avg_amount_paid,
        "total_amount_paid": total_amount_paid,
        "has_auto_renew": has_auto_renew,
        "has_cancelled": has_cancelled,
        "total_secs": total_secs,
        "num_unq": num_unq,
        "customer_tenure_days": customer_tenure_days,
        "listening_group": listening_group,
        "payment_variability": payment_variability
    }

----

# **3. LOAD MODEL + VALIDATE INPUT**

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

current_dir = os.getcwd()

while not os.path.exists(os.path.join(current_dir, "data", "processed")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/processed not found")
    current_dir = parent


os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/processed exists:", os.path.exists("data/processed"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


with open("model/final_churn_model.pkl", "rb") as f:
    model = pickle.load(f)

expected_features = [
    "gender", "age", "city_grouped", "registered_via_grouped",
    "avg_amount_paid", "total_amount_paid",
    "has_auto_renew", "has_cancelled",
    "total_secs", "num_unq",
    "customer_tenure_days", "listening_group",
    "payment_variability"
]

def validate_input(df):
    for col in expected_features:
        if col not in df.columns:
            df[col] = np.nan
    return df[expected_features]

Project root: c:\Users\pauli\OneDrive\Documentos\GitHub\customer-churn-intelligence-system
data/processed exists: True


----

# **4. CHURN PREDICTION + RISK LABEL**

In [3]:
def predict_churn(input_dict):
    df = pd.DataFrame([input_dict])
    df = validate_input(df)

    prob = model.predict_proba(df)[0][1]

    if prob < 0.10:
        risk = "Low"
    elif prob < 0.30:
        risk = "Medium"
    else:
        risk = "High"

    return prob, risk

---

# **5. BUSINESS INTELLIGENCE LAYER**

In [4]:
def compute_business_metrics(input_data, churn_prob):
    arpu = input_data.get("avg_amount_paid", 0)

    churn_rate = max(churn_prob, 0.01)
    ltv = arpu * (1 / churn_rate)

    revenue_at_risk = arpu * churn_prob
    potential_saved = revenue_at_risk * 0.10

    return {
        "ARPU": round(arpu, 2),
        "LTV": round(ltv, 2),
        "Revenue_at_Risk": round(revenue_at_risk, 2),
        "Potential_Revenue_Saved": round(potential_saved, 2)
    }

----

# **6. “WHY CHURN?” EXPLAINABILITY (RULE‑BASED)**

In [5]:
def explain_churn(input_data):
    explanations = []

    if input_data["has_cancelled"] == 1:
        explanations.append("Cancellation is the strongest churn driver.")

    if input_data["has_auto_renew"] == 0:
        explanations.append("No auto-renew indicates low commitment.")

    if input_data["listening_group"] in ["Low", "Medium-Low"]:
        explanations.append("Low engagement increases churn risk.")

    if input_data["payment_variability"] > 100:
        explanations.append("High payment variability suggests unstable subscription behavior.")

    if not explanations:
        explanations.append("Customer shows no major churn risk drivers.")

    return explanations

---

# **7. “WHEN WILL CHURN HAPPEN?” (TIMING ESTIMATE)**

In [6]:
def estimate_churn_timing(input_data):
    tenure = input_data["customer_tenure_days"]
    listening = input_data["listening_group"]

    if listening == "Low":
        return "Likely churn within 15–30 days."
    if listening == "Medium-Low":
        return "Likely churn within 30–45 days."
    if tenure < 200:
        return "Early-stage churn likely within 45 days."
    return "Churn risk exists but timing is uncertain."

---

# **8. LLM STRATEGIC RECOMMENDATION**

In [7]:
def llm_recommendation(prob, risk, explanations, business):
    text = f"""
Customer churn probability: {prob:.2%}
Risk level: {risk}

Key drivers:
- {'; '.join(explanations)}

Business impact:
- ARPU: €{business['ARPU']}
- LTV: €{business['LTV']}
- Revenue at risk: €{business['Revenue_at_Risk']}
- Potential revenue saved: €{business['Potential_Revenue_Saved']}

Based on this profile, suggest a retention strategy.
"""
    return text

---

# **9. FULL PIPELINE**

In [8]:
def score_customer(input_dict):
    prob, risk = predict_churn(input_dict)
    business = compute_business_metrics(input_dict, prob)
    explanations = explain_churn(input_dict)
    timing = estimate_churn_timing(input_dict)

    return {
        "probability": prob,
        "risk": risk,
        "business": business,
        "drivers": explanations,
        "timing": timing
    }

In [9]:
# 9.1 Prospective Churn & Revenue Forecast (Next 12 Months)

# Load full customer dataset for forecasting
df_full = pd.read_csv("data/processed/df_clean.csv")

# Score all customers
df_full["churn_prob"] = model.predict_proba(df_full[expected_features])[:, 1]

# Current ARPU (euros per user)
ARPU = df_full["avg_amount_paid"].mean()

# Forecast churners next year (customers)
expected_churners = int(df_full["churn_prob"].mean() * len(df_full))

# Revenue at risk (euros)
revenue_at_risk = expected_churners * ARPU

# Scenario modeling (formatted with units)
scenarios = pd.DataFrame({
    "Scenario": ["Base", "Optimistic", "Pessimistic"],
    "Expected_Churners (customers)": [
        expected_churners,
        int(expected_churners * 0.9),
        int(expected_churners * 1.2)
    ],
    "Revenue_at_Risk (€)": [
        round(revenue_at_risk, 2),
        round(revenue_at_risk * 0.9, 2),
        round(revenue_at_risk * 1.2, 2)
    ],
    "Revenue_Saved (€)": [
        round(revenue_at_risk * 0.10, 2),
        round(revenue_at_risk * 0.15, 2),
        round(revenue_at_risk * 0.05, 2)
    ]
})

# Convert to integers where appropriate
scenarios["Revenue_at_Risk (€)"] = scenarios["Revenue_at_Risk (€)"].astype(int)
scenarios["Revenue_Saved (€)"] = scenarios["Revenue_Saved (€)"].astype(int)

scenarios

c:\Users\pauli\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\pauli\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\pauli\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\pauli\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^

,Scenario,Expected_Churners (customers),Revenue_at_Risk (€),Revenue_Saved (€)
0,Base,87387,11433336,1143333
1,Optimistic,78648,10290002,1715000
2,Pessimistic,104864,13720003,571666


---

# **10. Streamlit Main App**

In [12]:
def run_app():
    st.title("KKBOX Churn Prediction & Retention Intelligence System")

    input_dict = ui_customer_input()

    if st.button("Predict Churn"):
        results = score_customer(input_dict)

        st.subheader("Churn Probability")
        st.metric("Probability", f"{results['probability']:.2%}")

        st.subheader("Risk Level")
        st.write(results["risk"])

        st.subheader("Why is this customer likely to churn?")
        for e in results["drivers"]:
            st.write("- " + e)

        st.subheader("When is churn likely?")
        st.write(results["timing"])

        st.subheader("Business Impact")
        st.json(results["business"])

        st.subheader("Strategic Recommendation (LLM)")
        st.write("→ Integrate your LLM here using the text prompt.")

        st.subheader("Prospective Churn & Revenue Forecast (Next 12 Months)")
        st.dataframe(scenarios)

In [13]:
sample_user = {
    "gender": "male",
    "age": 28,
    "city_grouped": "5",
    "registered_via_grouped": "3",
    "avg_amount_paid": 120,
    "total_amount_paid": 600,
    "has_auto_renew": 1,
    "has_cancelled": 0,
    "total_secs": 50000,
    "num_unq": 200,
    "customer_tenure_days": 800,
    "listening_group": "Medium-High",
    "payment_variability": 20
}

score_customer(sample_user)

{'probability': np.float64(0.9331870526137942),
 'risk': 'High',
 'business': {'ARPU': 120,
  'LTV': np.float64(128.59),
  'Revenue_at_Risk': np.float64(111.98),
  'Potential_Revenue_Saved': np.float64(11.2)},
 'drivers': ['Customer shows no major churn risk drivers.'],
 'timing': 'Churn risk exists but timing is uncertain.'}